In [0]:
from pyspark.sql import functions as F

CATALOG = "bucharest_mobility"
GOLD = "gold"
SILVER = "silver"

DELAY_TABLE = (
    f"{CATALOG}.{GOLD}.stop_delay_observations"
)

WEATHER_TABLE = (
    f"{CATALOG}.{SILVER}.weather_hourly"
)

TARGET_TABLE = (
    f"{CATALOG}.{GOLD}.route_weather_performance"
)

In [0]:
delays = spark.table(DELAY_TABLE)
weather = spark.table(WEATHER_TABLE)

print(f"Delay observations: {delays.count():,}")
print(f"Weather observations: {weather.count():,}")

In [0]:
delays_hourly = (
    delays
    .withColumn(
        "observation_hour",
        F.date_trunc(
            "hour",
            F.col("predicted_arrival"),
        ),
    )
)

In [0]:
route_hourly = (
    delays_hourly
    .groupBy(
        "route_id",
        "observation_hour",
    )
    .agg(
        F.count("*").alias(
            "prediction_count"
        ),

        F.countDistinct("trip_id").alias(
            "distinct_trips"
        ),

        F.round(
            F.avg("predicted_delay_seconds"),
            2,
        ).alias(
            "avg_predicted_delay_seconds"
        ),

        F.expr(
            """
            percentile_approx(
                predicted_delay_seconds,
                0.5
            )
            """
        ).alias(
            "median_predicted_delay_seconds"
        ),

        F.expr(
            """
            percentile_approx(
                predicted_delay_seconds,
                0.9
            )
            """
        ).alias(
            "p90_predicted_delay_seconds"
        ),

        F.round(
            F.avg(
                F.when(
                    F.col(
                        "predicted_delay_seconds"
                    ) > 300,
                    1.0,
                ).otherwise(0.0)
            ) * 100,
            2,
        ).alias(
            "pct_predictions_over_5min"
        ),
    )
)

print(
    f"Route-hour groups: "
    f"{route_hourly.count():,}"
)

In [0]:
weather_hourly = (
    weather
    .select(
        F.col("weather_timestamp").alias(
            "observation_hour"
        ),
        "temperature_2m",
        "precipitation",
        "rain",
        "snowfall",
        "weather_code",
        "wind_speed_10m",
    )
)

In [0]:
gold_df = (
    route_hourly
    .join(
        weather_hourly,
        on="observation_hour",
        how="left",
    )
)

In [0]:
total_groups = gold_df.count()

matched_weather = (
    gold_df
    .filter(
        F.col("temperature_2m").isNotNull()
    )
    .count()
)

print(
    f"Weather match: "
    f"{matched_weather:,}/{total_groups:,} "
    f"({matched_weather / total_groups * 100:.2f}%)"
)

In [0]:
duplicates = (
    gold_df
    .groupBy(
        "route_id",
        "observation_hour",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicates == 0, (
    f"route_weather_performance: "
    f"{duplicates} duplicate grains"
)

print(
    "✓ route_weather_performance: "
    "grain unique"
)

In [0]:
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(
    f"LOADED route_weather_performance: "
    f"{gold_df.count():,} rows"
)

In [0]:
final_gold = spark.table(TARGET_TABLE)

total = final_gold.count()

duplicates = (
    final_gold
    .groupBy(
        "route_id",
        "observation_hour",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_weather = (
    final_gold
    .filter(
        F.col("temperature_2m").isNull()
    )
    .count()
)

assert total > 0
assert duplicates == 0

weather_coverage = (
    (total - missing_weather) / total * 100
)

print(f"✓ rows: {total:,}")
print("✓ route × hour grain unique")
print(f"✓ weather coverage: {weather_coverage:.2f}%")
print("✓ route_weather_performance validated")